# Layer-wise Analysis of Harmfulness and Refusal Crystallization

This notebook implements comprehensive layer-wise analysis of where harmfulness and refusal concepts "crystallize" in transformer models.

## Research Questions

### 1. Layer-wise Dynamics (Priority #1)
- **Where do harmfulness and refusal representations crystallize?**
- Are there specific "critical layers" where these concepts form?
- Do early layers encode semantic harmfulness while late layers handle refusal decisions?
- Can we identify a "point of no return" layer after which interventions are ineffective?

### 2. Category-Specific Representations (Priority #3)
- **Is harmfulness category-specific or universal?**
- Do different harm categories (violence, illegal, hate speech) have distinct subspaces?
- Does a probe trained on one category generalize to others?
- Is refusal universal while harmfulness is category-specific?

### 3. Compositional Harmfulness (Priority #2)
- **How does context modulate harmfulness representations?**
- Does educational framing reduce harmfulness signal?
- Can we decompose harmfulness into base + context components?

## Setup

This notebook builds on `Harmfulness_and_Refusal_Probes.ipynb` and requires the `layerwise_analysis.py` module.

In [ ]:
# Standard imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import the layerwise analysis module
from layerwise_analysis import (
    analyze_layerwise_dynamics,
    analyze_category_specificity,
    create_compositional_examples,
    plot_layerwise_metrics,
    plot_category_analysis,
    save_analysis,
    load_analysis,
    train_layerwise_probes,
    compute_silhouette_scores
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load Cached Activations

First, load the compressed activations from the main notebook. You should have already run the data collection pipeline in `Harmfulness_and_Refusal_Probes.ipynb`.

In [ ]:
# Load your cached activations
# Replace with your actual cache file path
CACHE_FILE = "cache/llama3_1b_mixed_500.pkl"

with open(CACHE_FILE, 'rb') as f:
    compressed_data = pickle.load(f)

print(f"✓ Loaded {len(compressed_data)} examples")

# Quick sanity check
example = compressed_data[0]
print(f"\nExample structure:")
print(f"  ID: {example.example_id}")
print(f"  Label: {example.label}")
print(f"  Refused: {example.refused}")
print(f"  Number of layers: {len(example.acts_inst)}")
print(f"  Hidden dimension: {example.acts_inst[0].shape}")

# Count labels
n_harmful = sum(1 for ex in compressed_data if ex.label == "harmful")
n_harmless = len(compressed_data) - n_harmful
n_refused = sum(1 for ex in compressed_data if ex.refused)

print(f"\nDataset composition:")
print(f"  Harmful: {n_harmful}")
print(f"  Harmless: {n_harmless}")
print(f"  Refused: {n_refused} ({n_refused/len(compressed_data)*100:.1f}%)")

---

# Priority #1: Layer-wise Dynamics

## Where Do Harmfulness and Refusal Crystallize?

We'll train probes for all layers and analyze:
1. **Probe accuracy by layer** - Which layers have the strongest signal?
2. **Cluster separation** - Where do harmful/harmless examples separate?
3. **Direction stability** - Do directions change dramatically between layers?
4. **Critical layers** - When does each concept "crystallize"?

In [ ]:
# Run comprehensive layer-wise analysis
# This will train probes for all layers for both harmfulness and refusal

metrics = analyze_layerwise_dynamics(
    compressed_data,
    position="inst",  # Analyze at end-of-instruction position
    test_size=0.3,
    C=1.0  # Regularization strength
)

# Save results
save_analysis(metrics, "cache/layerwise_metrics_inst.pkl")

In [ ]:
# Visualize layer-wise dynamics
plot_layerwise_metrics(metrics, save_path="figures/layerwise_dynamics.png")

### Analysis: Early vs Middle vs Late Layers

Let's compare probe performance across layer groups:

In [ ]:
# Define layer groups
n_layers = len(metrics.layers)
early = range(0, n_layers // 3)
middle = range(n_layers // 3, 2 * n_layers // 3)
late = range(2 * n_layers // 3, n_layers)

def layer_group_stats(metric_array, layer_group):
    values = [metric_array[i] for i in layer_group]
    return {
        'mean': np.mean(values),
        'max': np.max(values),
        'argmax': list(layer_group)[np.argmax(values)]
    }

print("=" * 80)
print("LAYER GROUP ANALYSIS")
print("=" * 80)

print("\nHARMFULNESS PROBE ACCURACY:")
for name, group in [("Early", early), ("Middle", middle), ("Late", late)]:
    stats = layer_group_stats(metrics.harmfulness_acc, group)
    print(f"  {name:8s}: mean={stats['mean']:.3f}, max={stats['max']:.3f} (layer {stats['argmax']})")

print("\nREFUSAL PROBE ACCURACY:")
for name, group in [("Early", early), ("Middle", middle), ("Late", late)]:
    stats = layer_group_stats(metrics.refusal_acc, group)
    print(f"  {name:8s}: mean={stats['mean']:.3f}, max={stats['max']:.3f} (layer {stats['argmax']})")

print("\nHARMFULNESS SILHOUETTE SCORES:")
for name, group in [("Early", early), ("Middle", middle), ("Late", late)]:
    stats = layer_group_stats(metrics.harmfulness_silhouette, group)
    print(f"  {name:8s}: mean={stats['mean']:.3f}, max={stats['max']:.3f} (layer {stats['argmax']})")

print("\nREFUSAL SILHOUETTE SCORES:")
for name, group in [("Early", early), ("Middle", middle), ("Late", late)]:
    stats = layer_group_stats(metrics.refusal_silhouette, group)
    print(f"  {name:8s}: mean={stats['mean']:.3f}, max={stats['max']:.3f} (layer {stats['argmax']})")

### Identifying "Crystallization Points"

Find the first layer where probe accuracy crosses threshold (e.g., 70%, 80%, 90%):

In [ ]:
def find_crystallization_layer(accuracy_array, threshold):
    """Find first layer where accuracy exceeds threshold."""
    for i, acc in enumerate(accuracy_array):
        if acc >= threshold:
            return i
    return -1  # Never reached

print("=" * 80)
print("CRYSTALLIZATION POINTS")
print("=" * 80)

thresholds = [0.6, 0.7, 0.8, 0.9]

print("\nHarmfulness:")
for thresh in thresholds:
    layer = find_crystallization_layer(metrics.harmfulness_acc, thresh)
    if layer != -1:
        print(f"  {int(thresh*100)}% accuracy: Layer {layer}")
    else:
        print(f"  {int(thresh*100)}% accuracy: Never reached")

print("\nRefusal:")
for thresh in thresholds:
    layer = find_crystallization_layer(metrics.refusal_acc, thresh)
    if layer != -1:
        print(f"  {int(thresh*100)}% accuracy: Layer {layer}")
    else:
        print(f"  {int(thresh*100)}% accuracy: Never reached")

### Direction Stability Analysis

How similar are the probe directions across adjacent layers?

In [ ]:
# Plot adjacent layer similarity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Harmfulness
ax = axes[0]
adjacent_sim_harm = []
for i in range(len(metrics.layers) - 1):
    sim = metrics.harmfulness_direction_similarity[i, i+1]
    adjacent_sim_harm.append(sim)

ax.plot(range(len(adjacent_sim_harm)), adjacent_sim_harm, 'o-', linewidth=2)
ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='90% similarity')
ax.set_xlabel('Layer')
ax.set_ylabel('Cosine Similarity with Next Layer')
ax.set_title('Harmfulness Direction Stability')
ax.legend()
ax.grid(True, alpha=0.3)

# Refusal
ax = axes[1]
adjacent_sim_ref = []
for i in range(len(metrics.layers) - 1):
    sim = metrics.refusal_direction_similarity[i, i+1]
    adjacent_sim_ref.append(sim)

ax.plot(range(len(adjacent_sim_ref)), adjacent_sim_ref, 'o-', linewidth=2, color='orange')
ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='90% similarity')
ax.set_xlabel('Layer')
ax.set_ylabel('Cosine Similarity with Next Layer')
ax.set_title('Refusal Direction Stability')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/direction_stability.png', dpi=300, bbox_inches='tight')
plt.show()

# Find "jump" layers where direction changes significantly
print("\nLayers with significant direction changes (similarity < 0.85):")
print("\nHarmfulness:")
for i, sim in enumerate(adjacent_sim_harm):
    if sim < 0.85:
        print(f"  Layer {i} → {i+1}: similarity = {sim:.3f}")

print("\nRefusal:")
for i, sim in enumerate(adjacent_sim_ref):
    if sim < 0.85:
        print(f"  Layer {i} → {i+1}: similarity = {sim:.3f}")

### Key Findings Template

**Fill this in after running the analysis:**

1. **Critical Layers**:
   - Harmfulness crystallizes at layer: ___
   - Refusal crystallizes at layer: ___
   - Does harmfulness appear before refusal? ___

2. **Layer Groups**:
   - Harmfulness is strongest in ___ layers (early/middle/late)
   - Refusal is strongest in ___ layers (early/middle/late)

3. **Direction Stability**:
   - Harmfulness direction is (stable/unstable) across layers
   - Major changes occur at layers: ___
   - Refusal direction is (stable/unstable) across layers

4. **Interpretation**:
   - Evidence for/against "early semantic, late decision" hypothesis: ___
   - Potential "point of no return" layer: ___

---

# Priority #2: Compositional Harmfulness

## How Does Context Modulate Harmfulness?

We'll test how different contextual frames affect harmfulness representations:
- **Fiction frame**: "Write a fictional story about..."
- **Educational frame**: "Explain why someone might..."
- **Research frame**: "...for research purposes"
- **Prevention frame**: "How to prevent..."
- **Raw**: No context

**Hypothesis**: Context should reduce harmfulness signal but perhaps not eliminate it.

In [ ]:
# This section requires re-running the model on compositional examples
# For now, we'll outline the experiment structure

print("="*80)
print("COMPOSITIONAL HARMFULNESS EXPERIMENT")
print("="*80)
print("")
print("To run this experiment:")
print("")
print("1. Load your model (ChatModel from main notebook)")
print("2. Create compositional examples:")
print("")
print("   from layerwise_analysis import create_compositional_examples")
print("   harmful_texts = [ex.text for ex in loader.load_advbench(20)]")
print("   comp_examples = create_compositional_examples(harmful_texts)")
print("")
print("3. Run model on each compositional example")
print("4. Compare harmfulness probe scores across contexts")
print("")
print("Expected pattern:")
print("  Raw > Fiction > Research > Educational > Prevention")
print("")
print("Key questions:")
print("  - Does context reduce harmfulness signal or just refusal?")
print("  - Can we decompose: h_final = h_base + h_context?")
print("  - Which layers are most affected by context?")

---

# Priority #3: Category-Specific Representations

## Is Harmfulness Universal or Category-Specific?

We'll analyze whether different harm categories have distinct representations.

In [ ]:
# Pick the best harmfulness layer from Priority #1
best_harm_layer = np.argmax(metrics.harmfulness_acc)

print(f"Analyzing category specificity at layer {best_harm_layer}")
print(f"(Best harmfulness layer with acc={metrics.harmfulness_acc[best_harm_layer]:.3f})")

# Run category analysis
cat_analysis = analyze_category_specificity(
    compressed_data,
    layer=best_harm_layer,
    position="inst",
    test_size=0.3
)

if cat_analysis is not None:
    # Save results
    with open("cache/category_analysis.pkl", 'wb') as f:
        pickle.dump(cat_analysis, f)
    print("✓ Saved category analysis")

In [ ]:
# Visualize category analysis
if cat_analysis is not None:
    plot_category_analysis(cat_analysis, save_path="figures/category_analysis.png")

### Interpretation Guide

**Similarity Matrix**: High values (close to 1) = similar directions
- Diagonal is always 1 (category vs itself)
- Off-diagonal tells us if categories share harmfulness subspace
- **Hypothesis**: If harmfulness is universal, all values should be high (>0.8)
- **Hypothesis**: If category-specific, off-diagonal should be lower (<0.5)

**Generalization Matrix**: Accuracy when testing on different categories
- Diagonal = in-category accuracy (should be high)
- Off-diagonal = cross-category generalization
- **Good generalization** = high off-diagonal (>0.7)
- **Poor generalization** = low off-diagonal (<0.6)

**Key Findings**:
- If similarity is high but generalization is low → Directions point in similar directions but have different magnitudes
- If both are high → Universal harmfulness representation
- If both are low → Distinct category-specific subspaces

---

# Additional Experiments

## Compare t_inst vs t_postinst Positions

The paper analyzes two positions:
- **t_inst**: End of instruction (before assistant prefix)
- **t_postinst**: End of prompt (after assistant prefix)

Let's see how representations differ:

In [ ]:
# Run analysis for t_postinst position
print("Analyzing t_postinst position...")
metrics_postinst = analyze_layerwise_dynamics(
    compressed_data,
    position="postinst",
    test_size=0.3,
    C=1.0
)

save_analysis(metrics_postinst, "cache/layerwise_metrics_postinst.pkl")

In [ ]:
# Compare positions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Harmfulness accuracy
ax = axes[0, 0]
ax.plot(metrics.layers, metrics.harmfulness_acc, 'o-', label='t_inst', linewidth=2)
ax.plot(metrics_postinst.layers, metrics_postinst.harmfulness_acc, 's-', label='t_postinst', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Accuracy')
ax.set_title('Harmfulness Probe Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Refusal accuracy
ax = axes[0, 1]
ax.plot(metrics.layers, metrics.refusal_acc, 'o-', label='t_inst', linewidth=2)
ax.plot(metrics_postinst.layers, metrics_postinst.refusal_acc, 's-', label='t_postinst', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Accuracy')
ax.set_title('Refusal Probe Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Harmfulness silhouette
ax = axes[1, 0]
ax.plot(metrics.layers, metrics.harmfulness_silhouette, 'o-', label='t_inst', linewidth=2)
ax.plot(metrics_postinst.layers, metrics_postinst.harmfulness_silhouette, 's-', label='t_postinst', linewidth=2)
ax.set_xlabel('Layer')
ax.set_ylabel('Silhouette Score')
ax.set_title('Harmfulness Cluster Separation')
ax.legend()
ax.grid(True, alpha=0.3)

# Refusal silhouette
ax = axes[1, 1]
ax.plot(metrics.layers, metrics.refusal_silhouette, 'o-', label='t_inst', linewidth=2)
ax.plot(metrics_postinst.layers, metrics_postinst.refusal_silhouette, 's-', label='t_postinst', linewidth=2)
ax.set_xlabel('Layer')
ax.set_ylabel('Silhouette Score')
ax.set_title('Refusal Cluster Separation')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/position_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

---

# Summary and Conclusions

## Layer-wise Dynamics (Priority #1)

**Key Findings**:
1. Critical layers identified:
   - Harmfulness: Layer ___ (acc=___)
   - Refusal: Layer ___ (acc=___)

2. Temporal pattern:
   - [ ] Harmfulness appears before refusal (early semantic processing)
   - [ ] Both appear simultaneously
   - [ ] Refusal appears before harmfulness (unexpected!)

3. Direction stability:
   - Harmfulness: (stable/unstable)
   - Refusal: (stable/unstable)

**Implications**:
- For safety interventions: Target layers ___
- "Point of no return": Layer ___

## Category Specificity (Priority #3)

**Key Findings**:
1. Category similarity scores: ___
2. Cross-category generalization: ___
3. Conclusion: Harmfulness is (universal/category-specific)

**Implications**:
- Different categories (do/don't) need separate safety mechanisms

## Compositional Harmfulness (Priority #2)

**Key Findings**:
(To be filled after running compositional experiments)

---

## Next Steps

1. **Mechanism identification**: Use activation patching to find critical components
2. **Steering experiments**: Test intervention effectiveness by layer
3. **Cross-model analysis**: Do these patterns hold for other models?
4. **Temporal evolution**: Track dynamics during generation

## Paper Contributions

This analysis extends the harmfulness/refusal paper by:
1. ✅ Identifying where concepts crystallize (addresses stated limitation)
2. ✅ Quantifying category-specificity (paper mentions but doesn't deeply explore)
3. ✅ Analyzing compositional effects (novel contribution)
4. 📊 Providing layer-specific intervention targets (practical application)